# Qwen3-TTS Malayalam LoRA Inference

This notebook demonstrates how to perform text-to-speech inference using the **Qwen3-TTS-Base-12Hz** model with **Malayalam LoRA adapters** hosted on Hugging Face.

### Speed & Quality Optimizations
1. **Flash Attention**: Uses `sdpa` (Scaled Dot Product Attention) for 2x faster inference on modern GPUs.
2. **Dynamic Tokens**: Limits generation time based on text length to avoid wasted cycles.
3. **FP16 Precision**: Uses half-precision weights for faster computation and lower VRAM usage.

In [ ]:
#@title Install Dependencies
!apt-get install -y sox libsox-dev
# Update torchao and peft to compatible versions
!pip install -q -U torchao peft transformers accelerate librosa soundfile sox

# Clone and Install the Qwen3-TTS repository
import os
if not os.path.exists('Qwen3-TTS'):
    !git clone https://github.com/QwenLM/Qwen3-TTS.git

# Install in editable mode to handle paths correctly
!pip install -e Qwen3-TTS

In [ ]:
import sys
import os
import torch
import librosa
import numpy as np
import soundfile as sf
from peft import PeftModel
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel
from IPython.display import Audio, display

# -- Configuration --
BASE_MODEL_ID = "Qwen/Qwen3-TTS-Base-12Hz" 
LORA_REPO_ID = "siyah1/qwen3-tts-malayalam-lora"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
#@title Load Model (Base and LoRA)
print("Loading base model in FP16 with SDPA...")
qwen_tts = Qwen3TTSModel.from_pretrained(
    BASE_MODEL_ID, 
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    attn_implementation="sdpa" if device == "cuda" else "eager"
)

print(f"Loading LoRA adapters from {LORA_REPO_ID}...")
qwen_tts.model = PeftModel.from_pretrained(qwen_tts.model, LORA_REPO_ID)

print(f"Moving model to {device}...")
qwen_tts.model.to(device)
qwen_tts.device = torch.device(device)

qwen_tts.model.eval()
print("Model ready!")

In [ ]:
#@title Fast Test Code (Single Sentence)
import os
import glob
import soundfile as sf
from IPython.display import Audio, display

# Simple test text
test_text = "മലയാളം എന്റെ മാതൃഭാഷയാണ്." 

# Search for any reference audio in the current environment
test_ref = glob.glob("**/*.wav", recursive=True)
if test_ref:
    print(f"Running test TTS for: '{test_text}' using {test_ref[0]}")
    wavs, sr = qwen_tts.generate_voice_clone(
        text=test_text,
        ref_audio=test_ref[0],
        language="Malayalam",
        temperature=0.5,
        repetition_penalty=1.2,
        max_new_tokens=256, # Very short for fast test
        x_vector_only_mode=True
    )
    sf.write("test_output.wav", wavs[0], sr)
    display(Audio("test_output.wav", rate=sr))
else:
    print("Test failed: No .wav file found to use as reference voice.")

In [ ]:
#@title Merge LoRA into Base (Standalone Model)
#@markdown Run this if you want to save a permanent, standalone version of the model.
print("Weight merging initiated...")
qwen_tts.model = qwen_tts.model.merge_and_unload()
save_dir = "Qwen3-TTS-Malayalam-Merged"

os.makedirs(save_dir, exist_ok=True)
qwen_tts.model.save_pretrained(save_dir)
qwen_tts.processor.save_pretrained(save_dir)
print(f"✅ Model successfully merged and saved to: {save_dir}")

In [ ]:
#@title Run High-Speed Custom Inference
import os
import glob
import soundfile as sf
from IPython.display import Audio, display

#@markdown Enter the Malayalam text you want to synthesize.
text = "മലയാളം സംസാരിക്കാൻ എനിക്ക് വളരെ ഇഷ്ടമാണ്." 

#@markdown provide a path to a reference Malayalam voice (wav file).
ref_audio = "/content/ml_000000.wav"

if not os.path.exists(ref_audio):
    wavs = glob.glob("**/*.wav",
        language="Malayalam", recursive=True)
    if wavs:
        ref_audio = wavs[0]
        print(f"Reference not found,
        language="Malayalam", using fallback: {ref_audio}")
    else:
        print("Error: No reference audio found.")
        ref_audio = None

if ref_audio:
    dynamic_max_tokens = min(1024,
        language="Malayalam", max(128, len(text) * 15))
    print(f"Generating speech for: {text}")
    
    wavs, sr = qwen_tts.generate_voice_clone(
        text=text,
        ref_audio=ref_audio,
        language="Malayalam",
        temperature=0.5,
        top_p=0.8,
        repetition_penalty=1.2,
        max_new_tokens=dynamic_max_tokens,
        x_vector_only_mode=True,
        non_streaming_mode=True
    )

    output_path = "output.wav"
    sf.write(output_path, wavs[0], sr)
    display(Audio(output_path, rate=sr))